# Kataloga datu analīze
Fiktīvi dati. Ja nepieciešams, atkomentē instalāciju.

In [ ]:
# %pip install beautifulsoup4

In [ ]:
import csv
import json
from pathlib import Path
from bs4 import BeautifulSoup

In [ ]:
html = "<!doctype html>\n<html lang=\"lv\"><head><meta charset=\"utf-8\"><title>Grāmatu katalogs</title></head>\n<body><main id=\"catalog\"><h1>Grāmatu katalogs</h1><p>Fiktīvi mācību dati.</p>\n<article class=\"product\" id=\"book-1\">\n  <h2>Python pamati</h2>\n  <p class=\"category\">Programmēšana</p>\n  <p class=\"price\">25.90</p>\n  <a href=\"#book-1\">Apskatīt</a>\n</article>\n<article class=\"product\" id=\"book-2\">\n  <h2>Datu analīze</h2>\n  <p class=\"category\">Programmēšana</p>\n  <p class=\"price\">32.50</p>\n  <a href=\"#book-2\">Apskatīt</a>\n</article>\n<article class=\"product\" id=\"book-3\">\n  <h2>Latvijas vēsture</h2>\n  <p class=\"category\">Vēsture</p>\n  <p class=\"price\">18.00</p>\n  <a href=\"#book-3\">Apskatīt</a>\n</article>\n<article class=\"product\" id=\"book-4\">\n  <h2>Rīgas arhitektūra</h2>\n  <p class=\"category\">Arhitektūra</p>\n  <p class=\"price\">28.90</p>\n  <a href=\"#book-4\">Apskatīt</a>\n</article>\n<article class=\"product\" id=\"book-5\">\n  <h2>Algoritmu pamati</h2>\n  <p class=\"category\">Programmēšana</p>\n  <p class=\"price\">39.50</p>\n  <a href=\"#book-5\">Apskatīt</a>\n</article>\n<article class=\"product\" id=\"book-6\">\n  <h2>Ceļojumi pa Latviju</h2>\n  <p class=\"category\">Ceļošana</p>\n  <p class=\"price\">15.90</p>\n  <a href=\"#book-6\">Apskatīt</a>\n</article>\n</main></body></html>"

In [ ]:
soup = BeautifulSoup(html, "html.parser")

## No HTML uz sarakstu

In [ ]:
products = []
for item in soup.select("article.product"):
    name = item.select_one("h2").get_text(strip=True)
    price = float(item.select_one(".price").get_text(strip=True))
    products.append({"name": name, "price": price})
products

## Statistika

In [ ]:
prices = [row["price"] for row in products]
print(min(prices), max(prices), round(sum(prices)/len(prices), 2))

In [ ]:
cheap = sorted([row for row in products if row["price"] < 30], key=lambda row: row["price"])
cheap

## JSON

In [ ]:
output = Path("output")
output.mkdir(exist_ok=True)
with (output / "books.json").open("w", encoding="utf-8") as file:
    json.dump(products, file, ensure_ascii=False, indent=2)

In [ ]:
with (output / "books.json").open(encoding="utf-8") as file:
    assert json.load(file) == products

## CSV

In [ ]:
with (output / "books.csv").open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["name", "price"])
    writer.writeheader()
    writer.writerows(products)

In [ ]:
with (output / "books.csv").open(encoding="utf-8", newline="") as file:
    assert len(list(csv.DictReader(file))) == 6

## Dzīvokļu tabula

In [ ]:
apartments_html = "<!doctype html>\n<html lang=\"lv\"><head><meta charset=\"utf-8\"><title>Rīgas dzīvokļi</title></head>\n<body><h1>Rīgas dzīvokļi</h1><p>Fiktīvi sludinājumi.</p>\n<table id=\"apartments\"><caption>Īres piedāvājumi</caption><thead><tr><th scope=\"col\">Rajons</th><th scope=\"col\">Platība, m²</th><th scope=\"col\">Īre, EUR</th></tr></thead>\n<tbody><tr><td>Centrs</td><td>45</td><td>550</td></tr>\n<tr><td>Āgenskalns</td><td>60</td><td>650</td></tr>\n<tr><td>Teika</td><td>38</td><td>420</td></tr>\n<tr><td>Purvciems</td><td>52</td><td>480</td></tr>\n<tr><td>Mežaparks</td><td>70</td><td>750</td></tr></tbody></table></body></html>"
apt = BeautifulSoup(apartments_html, "html.parser")

In [ ]:
rents = []
for row in apt.select("tbody tr"):
    district, area, rent = [cell.get_text(strip=True) for cell in row.select("td")]
    rents.append(int(rent))
    print(district, round(int(rent)/int(area), 2))

In [ ]:
assert sum(rents)/len(rents) == 570

## Patstāvīgi
Atrodi lētāko dzīvokli un saglabā sarakstu JSON. Salīdzini rezultātu ar 6. un 7. nodarbības failu piemēriem.